# Week G / Chapter 7: From Single Channels to Timing Variability

Opening question: Why are spike times not perfectly identical even when the input looks similar?

This notebook is a runnable companion. It uses simple Python plots and calculations to reinforce the chapter without calculus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

dt = 0.1          # ms
T = 1000          # ms
t = np.arange(0, T, dt)

baseline = -70.0  # mV
threshold = -55.0 # mV
tau = 20.0        # ms
drive = 18.0      # arbitrary constant input

def run_noisy_neuron(noise_sd=1.2):
    v = np.full_like(t, baseline, dtype=float)
    spikes = []

    for i in range(1, len(t)):
        leak_term = (baseline - v[i-1]) / tau
        noise_term = rng.normal(0, noise_sd) * np.sqrt(dt)
        dv = (leak_term + drive / tau) * dt + noise_term
        v[i] = v[i-1] + dv

        if v[i] >= threshold:
            spikes.append(t[i])
            v[i] = baseline  # reset after spike

    spikes = np.array(spikes)
    isi = np.diff(spikes)
    rate_hz = len(spikes) / (T / 1000.0)
    cv = np.std(isi) / np.mean(isi) if len(isi) > 1 else np.nan
    return v, spikes, isi, rate_hz, cv

noise_levels = [0.6, 1.2, 2.0]
results = {}
for noise_sd in noise_levels:
    v, spikes, isi, rate_hz, cv = run_noisy_neuron(noise_sd=noise_sd)
    results[noise_sd] = {
        "v": v,
        "spikes": spikes,
        "isi": isi,
        "rate_hz": rate_hz,
        "cv": cv,
    }
    print(f"noise={noise_sd:.1f}  spikes={len(spikes):3d}  rate={rate_hz:6.2f} Hz  CV={cv:5.2f}")

fig, axes = plt.subplots(len(noise_levels), 1, figsize=(8, 6), sharex=True)
for ax, noise_sd in zip(axes, noise_levels):
    v = results[noise_sd]["v"]
    spikes = results[noise_sd]["spikes"]
    ax.plot(t, v, linewidth=0.8)
    ax.scatter(spikes, np.full_like(spikes, threshold), s=8)
    ax.axhline(threshold, linestyle="--", linewidth=1)
    ax.set_ylabel(f"noise {noise_sd}\nV_m (mV)")
axes[-1].set_xlabel("time (ms)")
fig.suptitle("Noisy threshold neuron: voltage traces and spike times")
fig.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.bar([str(noise_sd) for noise_sd in noise_levels], [results[n]["cv"] for n in noise_levels])
plt.xlabel("noise standard deviation")
plt.ylabel("ISI CV")
plt.title("Noise changes spike-time regularity")
plt.tight_layout()
plt.show()

## Student reflection

In 3-5 sentences, explain what the figure or calculation shows and how it connects to the chapter opening question.

## Teacher check

The notebook runs top-to-bottom, the figure has labeled axes when axes are used, and the student can explain the result without reading code aloud.